# Notebook 05: Paper Figures

**Single source of truth for all paper figures.**
Run top-to-bottom after notebooks 01-04. Never edit figure files manually.

| Figure | Description | Paper |
|--------|-------------|-------|
| fig1_circuit_diagram | Circuit schematic | Sec 3 |
| fig2_attention_heatmap | Attention pattern, top IS head | Sec 3 |
| fig3_induction_scores_baseline | Per-head IS, pretrained | Sec 4 |
| fig4_attribution_baseline | Attribution heatmap | Sec 4 |
| fig5_induction_code | IS over code fine-tuning | Sec 5 |
| fig6_induction_prose | IS over prose fine-tuning | Sec 5 |
| fig7_phase_transition | Phase transition detail | Sec 5 |
| fig8_adversarial | Adversarial probe results | Sec 5 |

In [ ]:
import sys; sys.path.insert(0, '..')
import json, numpy as np, torch, matplotlib.pyplot as plt
from pathlib import Path
from src.model.config import ModelConfig, EvalConfig
from src.model.train import load_pretrained_model, set_global_seed
from src.analysis.checkpoint_sweep import load_sweep_results
from src.analysis.phase_detection import detect_phase_transitions
from src.viz.attention_vis import *
from src.viz.circuit_diagram import plot_circuit_diagram, plot_attribution_heatmap

set_global_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
results_dir = Path('../experiments/results')
figures_dir = Path('../paper/figures'); figures_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# Load all saved results
baseline_ind = np.load(results_dir / 'baseline_induction_scores.npz')
baseline_attr = np.load(results_dir / 'baseline_attribution_scores.npz')
sweep_code = load_sweep_results(results_dir / 'sweep_code_seed42.npz')
sweep_prose = load_sweep_results(results_dir / 'sweep_prose_seed42.npz')
with open(results_dir / 'phase_transitions.json') as f: transitions_data = json.load(f)
adv_data = np.load(results_dir / 'adversarial_pre_post.npz', allow_pickle=True)
circuit_heads = [tuple(h) for h in baseline_attr['circuit_heads'].tolist()]
n_layers = baseline_ind['means'].shape[0]; n_heads = baseline_ind['means'].shape[1]
print('Circuit heads:', circuit_heads)

In [ ]:
# Fig 1: Circuit diagram
fig1 = plot_circuit_diagram(circuit_heads, n_layers=n_layers, n_heads=n_heads,
    title='Induction circuit (pretrained)', save_path=figures_dir/'fig1_circuit_diagram')
plt.show(); plt.close(fig1)

In [ ]:
# Fig 2: Attention heatmap of top induction head
model = load_pretrained_model(ModelConfig(), device=device)
torch.manual_seed(42)
seq_len = 10
prefix = torch.randint(0, model.cfg.d_vocab, (1, seq_len), device=device)
tokens = torch.cat([prefix, prefix], dim=1)
top = np.unravel_index(baseline_ind['means'].argmax(), baseline_ind['means'].shape)
top_layer, top_head = int(top[0]), int(top[1])
fig2 = plot_attention_heatmap(model, tokens, top_layer, top_head,
    title=f'Attention: L{top_layer}H{top_head} (top IS head)',
    save_path=figures_dir/'fig2_attention_heatmap', device=device)
plt.show(); plt.close(fig2)

In [ ]:
# Fig 3: Baseline IS heatmap
fig3 = plot_induction_score_heatmap(baseline_ind['means'],
    title='Per-head induction scores (pretrained)',
    save_path=figures_dir/'fig3_induction_scores_baseline')
plt.show(); plt.close(fig3)

In [ ]:
# Fig 4: Attribution heatmap
fig4 = plot_attribution_heatmap(baseline_attr['attribution_scores'], threshold=0.5,
    title='Attribution scores (pretrained)', save_path=figures_dir/'fig4_attribution_baseline')
plt.show(); plt.close(fig4)

In [ ]:
# Fig 5: IS over code fine-tuning
fig5 = plot_induction_score_over_training(sweep_code['steps'], sweep_code['induction_scores'],
    n_layers, n_heads, title='Induction score: Python code fine-tuning',
    highlight_heads=circuit_heads, save_path=figures_dir/'fig5_induction_code')
plt.show(); plt.close(fig5)

In [ ]:
# Fig 6: IS over prose fine-tuning
fig6 = plot_induction_score_over_training(sweep_prose['steps'], sweep_prose['induction_scores'],
    n_layers, n_heads, title='Induction score: TinyStories prose fine-tuning (control)',
    highlight_heads=circuit_heads, save_path=figures_dir/'fig6_induction_prose')
plt.show(); plt.close(fig6)

In [ ]:
# Fig 7: Phase transition for most-affected head
trans_code = transitions_data['code']
if trans_code:
    worst = max(trans_code, key=lambda t: abs(t['delta']))
    fig7 = plot_phase_transition(sweep_code['steps'], sweep_code['induction_scores'], trans_code,
        worst['layer'], worst['head'], save_path=figures_dir/'fig7_phase_transition')
    plt.show(); plt.close(fig7)
    print(f"Phase transition: L{worst['layer']}H{worst['head']} delta={worst['delta']:+.3f}")
else:
    print('No phase transitions detected - skipping Fig 7.')

In [ ]:
# Fig 8: Adversarial probe results
names = [str(n) for n in adv_data['names'].tolist()]
post_scores = {names[i]: float(adv_data['post'][i]) for i in range(len(names))}
fig8 = plot_adversarial_results(post_scores, title='Adversarial probe results (post fine-tuning)',
    save_path=figures_dir/'fig8_adversarial')
plt.show(); plt.close(fig8)
print('All figures saved to paper/figures/')